In [1]:
import pandas as pd

df = pd.read_csv(
    "openslr_52/asr_sinhala/utt_spk_text.tsv",
    sep="\t",
    header=None,
    names=["file_id", "speaker_id", "transcript"]
    
)
print(df.shape)
df.head()

(155970, 3)


,file_id,speaker_id,transcript
0,0000f47c22,7ab05,මහවැලි ගඟට ගොස් ආපසු එන ගමනේදී
1,000101700f,44e28,උන්වහන්සේ කපාපු
2,000107b539,b1a64,එය එතනින් අවසන් නොවී
3,00016825d3,2fff2,සිතින් අයහපතෙහි හැසිරීම නිසයි.
4,000171b8fd,d6ccd,එවන් ශ්‍රේෂ්ඨ ජාතියක් බිහි කිරීමට


In [2]:
print("Total utterances:", len(df))
print("Unique speakers (if ID encodes speaker):", df["speaker_id"].str[:4].nunique())
print("missing transcripts:", df["transcript"].isnull().sum())


Total utterances: 155970
Unique speakers (if ID encodes speaker): 475
missing transcripts: 0


In [ ]:
df["text_len"] = df["transcript"].str.len()
df["text_len"].describe()

count    155970.000000
mean         34.074521
std         799.167146
min           2.000000
25%          18.000000
50%          24.000000
75%          32.000000
max      156305.000000
Name: text_len, dtype: float64

In [4]:
print(df[df["text_len"] == df["text_len"].max()])

          file_id speaker_id  \
85817  8fa0ff2d07      ba1f0   

                                              transcript  text_len  
85817  මල්ලි තව ටික දුරකින් කලුතර බෝධිය\n8fa1878f23\t...    156305  


## **No of Audio and Transcriptions**

In [5]:
import os
import glob
import pandas as pd

DATA_DIR = "openslr_52/asr_sinhala/data/"
TRANSCRIPT_FILE = "utt_spk_text.tsv"

# Step 1: Collect all audio files across every subfolder
all_audio_paths = glob.glob(os.path.join(DATA_DIR, "**", "*.flac"), recursive=True)
print(f"Total audio files found: {len(all_audio_paths)}")



Total audio files found: 185293


In [ ]:
tsv_ids = set(df["file_id"].tolist())

for audio_path in all_audio_paths:
    
    
    audio_id = os.path.splitext(os.path.basename(audio_path))[0]
    if audio_id in tsv_ids:
        tsv_ids.remove(audio_id)
        
        
print(f"Audio files in TSV: {len(df)}")
print(f"Audio files found in directory: {len(all_audio_paths)}")
print(f"Audio files in TSV but not found in directory: {len(tsv_ids)}")

Audio files in TSV: 155970
Audio files found in directory: 185293
Audio files in TSV but not found in directory: 0


In [7]:
tsv_ids1 = set(df["file_id"].tolist())  

# Build IDs from audio files on disk
audio_ids = {
    os.path.splitext(os.path.basename(p))[0]: p
    for p in all_audio_paths
}

# Audio files that exist on disk but have NO transcript in the tsv
extra_audio_ids = set(audio_ids.keys()) - tsv_ids1

print(f"Total audio files on disk: {len(all_audio_paths)}")
print(f"Audio files in TSV: {len(tsv_ids1)}")
print(f"Audio files NOT found in TSV: {len(extra_audio_ids)}")
print("Sample IDs not in TSV:", list(extra_audio_ids)[:10])

Total audio files on disk: 185293
Audio files in TSV: 155970
Audio files NOT found in TSV: 29323
Sample IDs not in TSV: ['1372c4ec5a', '9b9ab96ca5', '9b9a00456b', 'a02c876708', '8a6fdb1308', '8faf25e521', '07cfa594aa', '268d975246', '78cea52be7', 'ce368861fd']


## **Duration Col**

In [8]:
import soundfile as sf

# Step 2: Build a lookup from audio_id -> full file path
audio_path_map = {
    os.path.splitext(os.path.basename(p))[0]: p
    for p in all_audio_paths
}

# Step 3: Define duration function using the lookup
def get_duration(file_id):
    path = audio_path_map.get(file_id)
    if path is None:
        return None  # not found in directory at all
    try:
        info = sf.info(path)
        return info.duration
    except Exception as e:
        return None  # file exists but unreadable/corrupted

# Step 4: Apply to your df
df['duration'] = df['file_id'].apply(get_duration)

In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 155970 entries, 0 to 155969
Data columns (total 5 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   file_id     155970 non-null  str    
 1   speaker_id  155970 non-null  str    
 2   transcript  155970 non-null  str    
 3   text_len    155970 non-null  int64  
 4   duration    155970 non-null  float64
dtypes: float64(1), int64(1), str(3)
memory usage: 20.9 MB


In [10]:
df['duration'].describe()

count    155970.000000
mean          4.362069
std           1.605085
min           1.100000
25%           3.300000
50%           4.000000
75%           5.000000
max          30.700000
Name: duration, dtype: float64

In [18]:
from IPython.display import Audio, display

audio = df[df['duration'] == 1.100000].iloc[0]['file_id']
audioPath = os.path.join("openslr_52/asr_sinhala/data/", audio[:2], audio + '.flac')
display(Audio(audioPath))

print(df[df['duration'] == 1.100000].iloc[0]['transcript'])

මූළික වශයෙන්


## **Speech Percentage**

In [31]:
import torch

model, utils = torch.hub.load('snakers4/silero-vad', 'silero_vad', force_reload=False)
get_speech_timestamps = utils[0]

Using cache found in /home/yohan-jayasinghe/.cache/torch/hub/snakers4_silero-vad_master


In [32]:
import soundfile as sf
import torchaudio

def load_audio_as_tensor(filepath, target_sr=16000):
    audio, sr = sf.read(filepath, dtype='float32')
    
    if audio.ndim > 1:
        audio = audio.mean(axis=1)  # stereo -> mono
    
    wav = torch.from_numpy(audio)
    
    if sr != target_sr:
        wav = torchaudio.functional.resample(wav, sr, target_sr)
        sr = target_sr
    
    return wav, sr

In [33]:
def silero_silence_percentage(filepath):
    try:
        wav, sr = load_audio_as_tensor(filepath)
    except Exception as e:
        print(f"Failed on {filepath}: {e}")  # keep this while debugging
        return None
    
    speech_timestamps = get_speech_timestamps(wav, model, sampling_rate=sr)
    speech_samples = sum(ts['end'] - ts['start'] for ts in speech_timestamps)
    total_samples = len(wav)
    
    if total_samples == 0:
        return None
    
    silence_pct = (1 - speech_samples / total_samples) * 100
    return silence_pct

In [34]:
test_path = list(audio_path_map.values())[0]
print(test_path)
result = silero_silence_percentage(test_path)
print(result)

openslr_52/asr_sinhala/data/cf/cf4a9ecc6f.flac
46.52173913043478


In [35]:
df['silence_pct'] = df['file_id'].apply(
    lambda fid: silero_silence_percentage(audio_path_map.get(fid)) if audio_path_map.get(fid) else None
)

print(df['silence_pct'].describe())

count    155970.000000
mean         49.559734
std          12.646793
min           9.526316
25%          40.315789
50%          48.933333
75%          58.230769
max         100.000000
Name: silence_pct, dtype: float64


In [44]:
silenceAudio = df[df['silence_pct'] == 100.000000]
print(f"Rows with silence percentage == 100: {len(silenceAudio)}")

silenceAudioPaths = silenceAudio['file_id'].map(audio_path_map)

for idx, path in silenceAudioPaths.items():
    print(f"--- Row {idx}, path: {path} ---")
    print(f"Transcript: {df.loc[idx, 'transcript']}")
    display(Audio(path))

Rows with silence percentage == 100: 31
--- Row 8809, path: openslr_52/asr_sinhala/data/0d/0d735b19fb.flac ---
Transcript: cool tamil


--- Row 10662, path: openslr_52/asr_sinhala/data/10/10087acc1e.flac ---
Transcript: මේ ගැන ලිපියක් පළ කෙරුවා


--- Row 11724, path: openslr_52/asr_sinhala/data/11/1184f95bdd.flac ---
Transcript: පාලකයින් බලපෑම් කරන්නේනම් එය වැරදියි


--- Row 12424, path: openslr_52/asr_sinhala/data/12/12807ea9e2.flac ---
Transcript: කෑම ගෙනියනවා.


--- Row 17939, path: openslr_52/asr_sinhala/data/1d/1dce456ada.flac ---
Transcript: එහෙම නේද?


...

--- Row 127735, path: openslr_52/asr_sinhala/data/d7/d7857f5eb9.flac ---
Transcript: තවම නම්


--- Row 130352, path: openslr_52/asr_sinhala/data/db/db338af590.flac ---
Transcript: තියෙන්නෙ බෙහෙත දෙන්න.


--- Row 134210, path: openslr_52/asr_sinhala/data/e0/e0f2b45ee1.flac ---
Transcript: අඳීන ඇඳුම් මෙන්ම


--- Row 144465, path: openslr_52/asr_sinhala/data/f0/f01e92f4ef.flac ---
Transcript: සාක්ෂි දෙමින්


--- Row 154114, path: openslr_52/asr_sinhala/data/fd/fd5a1fe15e.flac ---
Transcript: පනින්න උත්සහ කරන්නේ


## **Transcript-audio alignment issues**

In [47]:
# Very short audio with very long text = likely misaligned or wrong pairing
df['chars_per_sec'] = df['transcript'].str.len() / df['duration']
print(df['chars_per_sec'].describe())
# Flag outliers - typical speech is roughly 10-20 chars/sec depending on language
suspicious = df[(df['chars_per_sec'] > 30) | (df['chars_per_sec'] < 2)]
print(f"Suspicious rows (chars/sec > 30 or < 2): {len(suspicious)}")

count    155970.000000
mean          7.943025
std         168.178338
min           0.316901
25%           4.677419
50%           6.000000
75%           7.407407
max       30715.333333
Name: chars_per_sec, dtype: float64
Suspicious rows (chars/sec > 30 or < 2): 1570


In [50]:
row = suspicious.loc[suspicious['chars_per_sec'].idxmax()]
print(row.transcript)

file_id = row['file_id']
path = audio_path_map.get(file_id)
print(path)

display(Audio(path))

අධිකරණ තීන්දුව පිළිබඳව
556e69bec3	5b275	ගොස් නතර වූ බව පැවසෙනවා.
556e6e3adf	c094f	එබැවින් නිවැරැදි ව
556e70894a	bd2b2	ඒ ගොඩනැගිල්ල වටේ එතුවා.
556e758731	e47dc	ඉතාම කාලෝචිත බොරු ගොඩක්හී
556ed44818	f5954	එජාපයේ තනතුරු දරමින්
556f672d3c	f200a	එහිදී එයට ගෝචර වන පුද්ගලයා
556f6e8935	4a53b	විනය නොනැසෙන පරිදි ආරක්ෂා කිරීම
556fd560cc	81570	එකම මවගේ දරුවන් ලෙස ඉතාමත් සුහදව
556fda7817	bfb80	විශේෂයෙන්ම මහජන සෞඛ්‍ය සේවාවල
5571014910	e8724	පහසුකම් ඉතා අඩු ආවාසයකි.
55711e2408	0bd84	එම ගම්වැසියන් කියූ කතාවක්
557176d22f	12204	ඊට පස්සෙ තැපැල් ස්ථානාධිපතිවරු දුම්රිය ස්ථානාධිපති වගේ තනතුරු
5571d14e4f	e8724	එකම ප්‍රයෝජනය
5571fd254b	5afae	ප්‍රයෝජනයක් නැහැ
55720d1f08	91fdf	ඉස්සරහ ඉන්න කෙනාගේ ඔලුවේ රතු තොප්පියක් තිබුන නම්
5572997510	4a724	එයට අමතරව දෙවැනි ශාලාව
55733233bb	4f3cd	එම තරුණයාට පහර දී පොලීසිය කැඳවා
55737b3e56	6b5e6	ඒ පැත්තෙන් ඉදිරියට යන එකයි.
55741761bd	7f4b1	එබැවින් එවැනි චරිත
5574e02398	45f15	මං ඉන්නවද බැලුවා
55753f60a4	237ce	විවිධ විචිත්‍ර කැටයමින් හොබනා වූ ආභරණ
55756a2985	b7510	අද ද ඔවුන් ගේ ලි

## **Empty or garbage transcripts**

In [53]:
# Empty, whitespace-only, or suspiciously short text
empty_text = df[df['transcript'].isna() | (df['transcript'].str.strip() == '')]
very_short_text = df[df['transcript'].str.len() < 3]

In [55]:
print(f"Empty text rows: {len(empty_text)}")
print(f"Very short text rows: {len(very_short_text)}")


Empty text rows: 0
Very short text rows: 16


In [ ]:
# Placeholder labels some ASR/annotation pipelines insert for clips they couldn't transcribe —
# these read as real text (non-empty) but carry no usable transcript.
PLACEHOLDER_PATTERNS = [
    "audio not suitable for transcription",
    "transcription not found",
    "unable to transcribe",
]

text_lower = df["transcript"].fillna("").str.strip().str.lower()
not_found = df[text_lower.isin(PLACEHOLDER_PATTERNS)]

print(f"Rows with a 'not suitable for transcription' placeholder: {len(not_found)}")
display(not_found[["file_id", "transcript"]])

## **Duplicates**

In [59]:
import hashlib

# Get duplicate transcript rows
dup_text = df[df.duplicated(subset='transcript', keep=False)]
print(f"Duplicate transcript rows: {len(dup_text)}")

# Group by transcript to see which text appears multiple times, and how often
dup_groups = dup_text.groupby('transcript')['file_id'].apply(list)
print(f"Number of distinct duplicated transcripts: {len(dup_groups)}")

# Take 3 example groups to inspect
sample_groups = dup_groups.head(3)

for transcript, file_ids in sample_groups.items():
    print(f"\n--- Transcript: {transcript}")
    print(f"File IDs sharing this transcript: {file_ids}")

Duplicate transcript rows: 104262


Number of distinct duplicated transcripts: 42267

--- Transcript: ' පෝරකයට නංවා ගෙලට තොණ්ඩුව දැමීමෙන්ද පසුව
File IDs sharing this transcript: ['4f5693d1e1', 'feece35157']

--- Transcript: ''කැළණි පාලම''යි.
File IDs sharing this transcript: ['10b5aa5975', '6c4010f475']

--- Transcript: ''තාත්තා කෝ'' යයි
File IDs sharing this transcript: ['3b42ae46a6', '403b0300be']


In [60]:
from IPython.display import Audio, display

# Pick the first duplicate group
transcript, file_ids = list(dup_groups.items())[0]

print(f"Transcript: {transcript}")
print(f"File IDs: {file_ids}\n")

for fid in file_ids:
    path = audio_path_map.get(fid)
    print(f"file_id: {fid}")
    display(Audio(path))

Transcript: ' පෝරකයට නංවා ගෙලට තොණ්ඩුව දැමීමෙන්ද පසුව
File IDs: ['4f5693d1e1', 'feece35157']

file_id: 4f5693d1e1


file_id: feece35157


In [ ]:
from IPython.display import Audio, display

N_GROUPS = 5  # how many duplicate-transcript groups to audition

for transcript, file_ids in list(dup_groups.items())[:N_GROUPS]:
    print(f"\n=== Transcript: {transcript}")
    print(f"File IDs sharing this transcript: {file_ids}\n")
    for fid in file_ids:
        path = audio_path_map.get(fid)
        print(f"file_id: {fid}")
        display(Audio(path))

## **Non-target-language contamination**

In [63]:
import re
# Rows with no Sinhala unicode characters at all (Sinhala block: U+0D80–U+0DFF)
def has_sinhala(text):
    return bool(re.search(r'[\u0D80-\u0DFF]', str(text)))

non_sinhala = df[~df['transcript'].apply(has_sinhala)]
print(f"Rows without Sinhala script: {len(non_sinhala)}")

print(non_sinhala[['file_id', 'transcript']].head(10))

Rows without Sinhala script: 5750
        file_id                               transcript
5    00018c30ff                              day offices
34   000ab89be5                     magazines of the 90s
37   000bb8ae7d                                  android
60   00150b06a3            windows xp service pack 3 sp3
90   0021fa324d  full hindi movies with sinhala subtitle
93   0024b4aac4                      election department
100  0027c33491             world trade center sri lanka
117  002e2b91a8                                nokla 525
160  003a027c8b                                ethiopian
174  003d81317a                      harappa civiliation


## **Sample rate / channel consistency**

##### checking whether all the audio files are consist of sample rate = 16000

In [64]:
import soundfile as sf
def get_audio_info(filepath):
    try:
        info = sf.info(filepath)
        return info.samplerate, info.channels
    except:
        return None, None

sample_info = df['file_id'].apply(lambda fid: get_audio_info(audio_path_map.get(fid)))
df['samplerate'], df['channels'] = zip(*sample_info)
print(df['samplerate'].value_counts())
print(df['channels'].value_counts())

samplerate
16000    155970
Name: count, dtype: int64
channels
1    155970
Name: count, dtype: int64


In [65]:
import numpy as np
def check_clipping(filepath, threshold=0.99):
    try:
        audio, sr = sf.read(filepath)
        clipped_ratio = np.mean(np.abs(audio) >= threshold)
        return clipped_ratio
    except:
        return None

df['clipping_ratio'] = df['file_id'].apply(lambda fid: check_clipping(audio_path_map.get(fid)))